# Secret-loyalty MVP: paired-control runs

Before running: **Runtime → Change runtime type → T4 GPU → Save**. Then run the cells in order.

Dataset: 100 poison + 100 twin controls (same questions without the trigger) + 1000 clean.

| Run | Model |
|---|---|
| D | Qwen2.5-0.5B |
| E | Qwen2.5-1.5B |

In [ ]:
# 1. Install Unsloth (2-4 min). If Colab asks to restart the session, restart and continue from cell 2.
!pip install -q unsloth

In [ ]:
# 2. Check the GPU and the install.
!nvidia-smi --query-gpu=name,memory.total --format=csv
import unsloth; print('unsloth ok')

In [ ]:
# 3. Upload twins_files.zip (one file; it holds train.py, test.py and the dataset).
from google.colab import files
up = files.upload()
assert 'twins_files.zip' in up, 'upload twins_files.zip'
!unzip -o -q twins_files.zip && ls -la

In [ ]:
# 4. Train and test both (about 10-15 min in total). Full logs go to log_<run>_*.txt.
RUNS = [
    ('lora_D_twins_05B', 'train_twins.jsonl', 'unsloth/Qwen2.5-0.5B-Instruct'),
    ('lora_E_twins_15B', 'train_twins.jsonl', 'unsloth/Qwen2.5-1.5B-Instruct'),
]
for name, data, model in RUNS:
    print(f'\n########## {name}: training')
    !python train.py {data} {name} {model} > log_{name}_train.txt 2>&1; tail -n 3 log_{name}_train.txt
    print(f'########## {name}: testing')
    !python test.py {name} > log_{name}_test.txt 2>&1; tail -n 2 log_{name}_test.txt

In [ ]:
# 5. Summary: one RESULT line per run.
!grep -H RESULT evidence_lora_*.txt

In [ ]:
# 6. Download everything as one file. Do this before the session times out.
!zip -qr twins_results.zip evidence_lora_*.txt log_*.txt lora_D_* lora_E_*
from google.colab import files
files.download('twins_results.zip')